# Part A

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(42)

names = ["Aridia",
         "Borland",
         "Cyrenia",
         "Deltora",
         "Estovia",
         "Freland",
         "Galenia",
         "Hesperia"
        ]

A_full = torch.tensor([
    #A  B  C  D  E  F  G  H
    [0, 1, 1, 0, 0, 0, 1, 0], # A
    [1, 0, 0, 0, 0, 1, 1, 0], # B
    [1, 0, 0, 0, 0, 1, 1, 0], # C
    [0, 0, 0, 0, 1, 1, 0, 1], # D
    [0, 0, 0, 1, 0, 0, 0, 1], # E
    [0, 1, 1, 1, 0, 0, 1, 1], # F
    [1, 1, 1, 0, 0, 1, 0, 0], # G
    [0, 0, 0, 1, 1, 1, 0, 0] # H
    ], dtype=torch.float32
)

degree = A_full.sum(dim=1)
degree_ave = torch.mean(degree)
unique_edges = torch.sum(degree)/2

X = torch.tensor([
    [5.8, 3.1, 4.2, 46, 6.2],
    [5.1, 3.8, 5.0, 42, 5.7],
    [6.4, 2.7, 3.8, 51, 7.1],
    [3.2, 6.5, 7.4, 30, 3.1],
    [2.8, 7.2, 8.1, 27, 2.8],
    [4.4, 4.1, 5.6, 39, 4.9],
    [5.6, 3.0, 4.5, 48, 6.5],
    [3.5, 5.8, 6.9, 33, 3.6]
    ], dtype=torch.float32
)

X_mean = X.mean(dim=0)
X_std = X.std(dim=0)
X_scaled = (X - X_mean)/X_std

print("=== Part A: Graph and features ===")
# checking the shapes first, X should be 8 countries by 5 features and A should be 8 by 8
print(f"X shape: {X.shape}")
print(f"A_full shape: {A_full.shape}")
# A_full is symmetric if flipping it (A_full.T) gives back the exact same matrix
print(f"Symmetric: {torch.equal(A_full, A_full.T)}")
print(f"Diagonal: {torch.diag(A_full).numpy()}")
print(f"Degree of Country: {degree.numpy()}")
print(f"Mean Degree: {degree_ave.numpy()}")
print(f"Unique Edges: {unique_edges.numpy()}")

=== Part A: Graph and features ===
X shape: torch.Size([8, 5])
A_full shape: torch.Size([8, 8])
Symmetric: True
Diagonal: [0. 0. 0. 0. 0. 0. 0. 0.]
Degree of Country: [3. 3. 3. 3. 2. 5. 4. 3.]
Mean Degree: 3.25
Unique Edges: 13.0


Summing all entries of a symmetric adjacency matrix double-counts undirected edges because the connection is counted on both sides. For example, A--B, B is counted on row A and A is counted on row B which essentially double counts the same connection.

Standardizing the feature matrix is important because each feature can have different scales so larger data in a feature will give that feature a higher weight and standardizing will remove that problem making each feature equally important.

# Part B

In [2]:
# B1
A_train = torch.tensor([
    #A  B  C  D  E  F  G  H
    [0, 1, 1, 0, 0, 0, 0, 0], # A
    [1, 0, 0, 0, 0, 1, 1, 0], # B
    [1, 0, 0, 0, 0, 1, 1, 0], # C
    [0, 0, 0, 0, 1, 1, 0, 0], # D
    [0, 0, 0, 1, 0, 0, 0, 1], # E
    [0, 1, 1, 1, 0, 0, 1, 1], # F
    [0, 1, 1, 0, 0, 1, 0, 0], # G
    [0, 0, 0, 0, 1, 1, 0, 0] # H
    ], dtype=torch.float32
)

print("\n=== B1: Training graph ===")
print(f"A_train shape: {A_train.shape}")
train_degree = A_train.sum(dim=1)
train_unique_edges = torch.sum(train_degree)/2
print(f"Training Unique Edges: {train_unique_edges.numpy()}")
print(f"Training graph contains 11 unique edges: {train_unique_edges.numpy() == 11.0}")



=== B1: Training graph ===
A_train shape: torch.Size([8, 8])
Training Unique Edges: 11.0
Training graph contains 11 unique edges: True


In [3]:
# B2
def aggregate(A, H):
    I = torch.eye(A.shape[0])
    A_hat = A + I
    degree_hat = A_hat.sum(dim=1)
    degree_hat = degree_hat.clamp(min=1) # handle degrees safely, basically does max(degree_hat, 1)
    D_inverse = torch.diag(1.0 / degree_hat)
    X_aggregated = D_inverse @ A_hat @ H
    assert X_aggregated.shape == H.shape, "X_aggregated's shape == H.shape"
    return X_aggregated

In [4]:
# B3
class GNNEncoder(nn.Module):
    def __init__(self, input_features=5, hidden_size=8, embedding_size=3):
        super().__init__()
        self.linear_one = nn.Linear(input_features, hidden_size)
        self.activation = nn.ReLU()
        self.linear_two = nn.Linear(hidden_size, embedding_size)

    def forward(self, A, X):
        layer_one = self.linear_one(aggregate(A, X))
        X_activated = self.activation(layer_one)
        layer_two = self.linear_two(aggregate(A, X_activated))
        return layer_two

class GraphAutoencoder(nn.Module):
    def __init__(self, input_features_size, hidden_size, embedding_size):
        super().__init__()
        self.encoder = GNNEncoder(input_features_size, hidden_size, embedding_size)

    def forward(self, A, X):
        Z = self.encoder(A, X)
        scores = Z @ Z.T
        A_reconstructed = torch.sigmoid(scores)
        return A_reconstructed, Z


def report(model, A, X):
    model.eval()
    with torch.no_grad():
        probabilities, embeddings = model(A, X)

    print("Reconstructed edge probabilities (diagonal NOT forced to zero):")
    print(probabilities)

    print("\nCountry: Embedding")
    for i, embedding in enumerate(embeddings):
        print(f"{names[i]}: {embedding}")

    return probabilities, embeddings



In [5]:
# B4
torch.manual_seed(42)
INPUT_FEATURES = X_scaled
ENCODER_GRAPH = A_train
BASELINE_RECONSTRUCTION_TARGET = A_train
HIDDEN_DIMENSION = 8
EMBEDDING_DIMENSION = 3
LEARNING_RATE = 0.01
EPOCHS = 2000

model = GraphAutoencoder(INPUT_FEATURES.shape[1], HIDDEN_DIMENSION, EMBEDDING_DIMENSION)
loss_function = nn.BCELoss()
optimizer = torch.optim.Adam(
    model.parameters(), lr=LEARNING_RATE
)

print("\n=== B4: Training (mean aggregation) ===")
for epoch in range(EPOCHS):
    optimizer.zero_grad()
    A_reconstructed, Z = model(ENCODER_GRAPH, INPUT_FEATURES)
    loss = loss_function(A_reconstructed, BASELINE_RECONSTRUCTION_TARGET)
    loss.backward()
    optimizer.step()

    if epoch % 200 == 0:
        print(f"Epoch {epoch}: loss = {loss}")

    if epoch == EPOCHS - 1:
        print(f"Final Loss: {loss}\n")

print(f"Z shape: {Z.shape}")
report(model, ENCODER_GRAPH, INPUT_FEATURES)


=== B4: Training (mean aggregation) ===
Epoch 0: loss = 0.7324681282043457
Epoch 200: loss = 0.599016547203064
Epoch 400: loss = 0.5987579226493835
Epoch 600: loss = 0.5982092618942261
Epoch 800: loss = 0.5979307293891907
Epoch 1000: loss = 0.5976467728614807
Epoch 1200: loss = 0.5973795652389526
Epoch 1400: loss = 0.5971369743347168
Epoch 1600: loss = 0.5967096090316772
Epoch 1800: loss = 0.5961896181106567


Final Loss: 0.5949258804321289

Z shape: torch.Size([8, 3])
Reconstructed edge probabilities (diagonal NOT forced to zero):
tensor([[0.7125, 0.7048, 0.7126, 0.2945, 0.2021, 0.5682, 0.7180, 0.3176],
        [0.7048, 0.7193, 0.7195, 0.2923, 0.1910, 0.5901, 0.7418, 0.3232],
        [0.7126, 0.7195, 0.7223, 0.2880, 0.1887, 0.5847, 0.7391, 0.3166],
        [0.2945, 0.2923, 0.2880, 0.7030, 0.7986, 0.4234, 0.2750, 0.6767],
        [0.2021, 0.1910, 0.1887, 0.7986, 0.9028, 0.3691, 0.1674, 0.7621],
        [0.5682, 0.5901, 0.5847, 0.4234, 0.3691, 0.5446, 0.6067, 0.4410],
        [0.7180, 0.7418, 0.7391, 0.2750, 0.1674, 0.6067, 0.7685, 0.3107],
        [0.3176, 0.3232, 0.3166, 0.6767, 0.7621, 0.4410, 0.3107, 0.6558]])

Country: Embedding
Aridia: tensor([ 0.8484, -0.3442, -0.2634])
Borland: tensor([ 0.9591, -0.1435, -0.0263])
Cyrenia: tensor([ 0.9458, -0.2209, -0.1128])
Deltora: tensor([-0.8803,  0.2511,  0.1533])
Estovia: tensor([-1.4574,  0.2959,  0.1316])
Freland: tensor([0.3941, 0.0685, 0.1370

(tensor([[0.7125, 0.7048, 0.7126, 0.2945, 0.2021, 0.5682, 0.7180, 0.3176],
         [0.7048, 0.7193, 0.7195, 0.2923, 0.1910, 0.5901, 0.7418, 0.3232],
         [0.7126, 0.7195, 0.7223, 0.2880, 0.1887, 0.5847, 0.7391, 0.3166],
         [0.2945, 0.2923, 0.2880, 0.7030, 0.7986, 0.4234, 0.2750, 0.6767],
         [0.2021, 0.1910, 0.1887, 0.7986, 0.9028, 0.3691, 0.1674, 0.7621],
         [0.5682, 0.5901, 0.5847, 0.4234, 0.3691, 0.5446, 0.6067, 0.4410],
         [0.7180, 0.7418, 0.7391, 0.2750, 0.1674, 0.6067, 0.7685, 0.3107],
         [0.3176, 0.3232, 0.3166, 0.6767, 0.7621, 0.4410, 0.3107, 0.6558]]),
 tensor([[ 0.8484, -0.3442, -0.2634],
         [ 0.9591, -0.1435, -0.0263],
         [ 0.9458, -0.2209, -0.1128],
         [-0.8803,  0.2511,  0.1533],
         [-1.4574,  0.2959,  0.1316],
         [ 0.3941,  0.0685,  0.1370],
         [ 1.0905, -0.0797,  0.0675],
         [-0.7231,  0.2802,  0.2082]]))

# Part C

In [6]:
# C1
print("\n=== C1: Hidden-link comparisons ===")
aridia, borland, cyrenia, deltora, estovia, freland, galenia, hesperia = range(8)

with torch.no_grad():
    _, Z = model(ENCODER_GRAPH, INPUT_FEATURES)

    aridia_galenia = torch.sigmoid(Z[aridia] @ Z[galenia])
    aridia_estovia = torch.sigmoid(Z[aridia] @ Z[estovia])
    deltora_hesperia = torch.sigmoid(Z[deltora] @ Z[hesperia])
    deltora_cyrenia = torch.sigmoid(Z[deltora] @ Z[cyrenia])

    print(f"Aridia - Galenia: {aridia_galenia}")
    print(f"Aridia - Estovia: {aridia_estovia}")
    print(f"Hidden real link outranks non-edge: {aridia_galenia > aridia_estovia}\n")

    print(f"Deltora - Hesperia: {deltora_hesperia}")
    print(f"Deltora - Cyrenia: {deltora_cyrenia}")
    print(f"Hidden real link outranks non-edge: {deltora_hesperia > deltora_cyrenia}")



=== C1: Hidden-link comparisons ===
Aridia - Galenia: 0.7180455327033997
Aridia - Estovia: 0.20214448869228363
Hidden real link outranks non-edge: True

Deltora - Hesperia: 0.6767332553863525
Deltora - Cyrenia: 0.28795886039733887
Hidden real link outranks non-edge: True


In [7]:
# C2
print("\n=== C2: Candidate ranking ===")

def unique_pairs(A):
    # Undirected edges occur twice in A. Score each pair once, excluding the
    # diagonal: self-loops support aggregation but are not prediction targets.
    return torch.triu_indices(
        A.shape[0], A.shape[1], offset=1,
        device=A.device,
    )

rows, cols = unique_pairs(A_train)
reconstructed_scores = []
with torch.no_grad():
    probabilities, Z = model(ENCODER_GRAPH, INPUT_FEATURES)

    # Because pairs were appended in index order and because Python's sorting is stable,
    # pairs with tied scores stay in increasing pair index order.
    for i, j in zip(rows.tolist(), cols.tolist()):
        if A_train[i, j] == 0:
            reconstructed_scores.append((probabilities[i, j], i, j))
    reconstructed_scores = sorted(reconstructed_scores, key=lambda x:x[0], reverse=True)

for idx, score in enumerate(reconstructed_scores):
    if score[1] == aridia and score[2] == galenia:
        print(f"Aridia - Galenia is Rank {idx + 1}")
    if score[1] == deltora and score[2] == hesperia:
        print(f"Deltora - Hesperia is Rank {idx + 1}\n")

for i in range(5):
    print(f"{names[reconstructed_scores[i][1]]} - {names[reconstructed_scores[i][2]]}: {reconstructed_scores[i][0]}")



=== C2: Candidate ranking ===
Aridia - Galenia is Rank 2
Deltora - Hesperia is Rank 3

Borland - Cyrenia: 0.7194586992263794
Aridia - Galenia: 0.7180455327033997
Deltora - Hesperia: 0.6767332553863525
Aridia - Freland: 0.5682361721992493
Estovia - Freland: 0.36911439895629883


In [8]:
# C3
print("\n=== C3: Nearest neighbors and anomaly ranking ===")
country_to_closest_country = {}

with torch.no_grad():
    _, Z = model(ENCODER_GRAPH, INPUT_FEATURES)
    # https://docs.pytorch.org/docs/main/generated/torch.cdist.html
    distances = torch.cdist(Z, Z)

    for i in range(len(distances)):
        closest = float('inf')
        for j in range(len(distances[i])):
            if i == j: continue
            if distances[i, j] < closest:
                closest = distances[i, j]
                country_to_closest_country[i] = (j, closest)

for country in country_to_closest_country:
    print(f"The closest country to {names[country]} is {names[country_to_closest_country[country][0]]}. Distance: {country_to_closest_country[country][1]:.4f}")


with torch.no_grad():
    _, Z = model(ENCODER_GRAPH, INPUT_FEATURES)

    centroid = Z.mean(dim=0)

    centroid_distances = torch.norm(Z - centroid, dim=1)

# rank descending (largest distance = most "anomalous");
# stable sort so ties resolve in country-index order
anomaly_ranking = sorted(
    range(len(names)), key=lambda i: centroid_distances[i].item(), reverse=True
)

print(f"\nEmbedding centroid: {centroid}\n")
print("Rank  Country     Distance to centroid")
for rank, i in enumerate(anomaly_ranking, start=1):
    print(f"{rank:<5} {names[i]:<11} {centroid_distances[i]:.4f}")






=== C3: Nearest neighbors and anomaly ranking ===
The closest country to Aridia is Cyrenia. Distance: 0.2177
The closest country to Borland is Cyrenia. Distance: 0.1168
The closest country to Cyrenia is Borland. Distance: 0.1168
The closest country to Deltora is Hesperia. Distance: 0.1691
The closest country to Estovia is Deltora. Distance: 0.5792
The closest country to Freland is Borland. Distance: 0.6252
The closest country to Galenia is Borland. Distance: 0.1736
The closest country to Hesperia is Deltora. Distance: 0.1691

Embedding centroid: tensor([0.1471, 0.0134, 0.0369])

Rank  Country     Distance to centroid
1     Estovia     1.6319
2     Deltora     1.0610
3     Galenia     0.9484
4     Hesperia    0.9262
5     Cyrenia     0.8457
6     Aridia      0.8425
7     Borland     0.8294
8     Freland     0.2721


The centroid distance shows the countries the model represents differently from the rest. It is a pattern in the embedding space, shaped by the objective of the training, the sizes of the clusters, and the overall structure of the network. This heuristic alone does not contain real economic criteria to call a country "economically abnormal."

### Part D

In [9]:
# D1
print("\n=== D1: Borland inflation shock ===")

X_changed = X.clone()
X_changed[borland, 1] = 12.0

X_changed_scaled = (X_changed - X_mean)/X_std

with torch.no_grad():
    Z_before = model.encoder(A_train, X_scaled)
    Z_after  = model.encoder(A_train, X_changed_scaled)

    print("Country: ||Z_after - Z_before||_2")
    for c in [borland, aridia, freland, galenia, estovia]:
        change = torch.norm(Z_after[c] - Z_before[c])
        print(f"{names[c]:>7}: {change:.4f}")



=== D1: Borland inflation shock ===
Country: ||Z_after - Z_before||_2
Borland: 4.3650
 Aridia: 2.8691
Freland: 2.1960
Galenia: 3.2940
Estovia: 0.0000


Only Borland's inflation was changed but the embeddings of other countries also moved. Each GNN layer averages a country's features with its trading partners so the change in Borland spreads through the links in A_train. Borland changed the most (4.37) since the change is in its own features. Its direct neighbors Galenia (3.29), Aridia (2.87), and Freland (2.20) changed as well, but Freland changed the least out of the three because it has 5 trading partners so Borland's features get averaged out more. Estovia did not change at all (0.0) because it is 3 hops away from Borland (Borland -> Freland -> Deltora -> Estovia) and a 2 layer GNN can only reach 2 hops. This is also why a non-neighbor can still change if it is 2 hops away, for example Cyrenia through Aridia.

In [10]:
# D2
print("\n=== D2: Feature-only baseline ===")
class BaselineEncoder(nn.Module):
    def __init__(self, input_features=5, hidden_size=8, embedding_size=3):
        super().__init__()
        self.linear_one = nn.Linear(input_features, hidden_size)
        self.activation = nn.ReLU()
        self.linear_two = nn.Linear(hidden_size, embedding_size)

    def forward(self, X):
        return self.linear_two(self.activation(self.linear_one(X)))

torch.manual_seed(42)
baseline = BaselineEncoder(INPUT_FEATURES.shape[1], HIDDEN_DIMENSION, EMBEDDING_DIMENSION)

with torch.no_grad():
    Z_baseline = baseline(X_scaled)
    print(f"Baseline embedding shape: {Z_baseline.shape}")




=== D2: Feature-only baseline ===
Baseline embedding shape: torch.Size([8, 3])


The BaselineEncoder only receives X_scaled so each country is represented by its own 5 features only. The GNN, on the other hand, also has the adjacency matrix A_train, which lets it average each country's features with its trading partners, and with 2 layers it can reach countries up to 2 hops away. The baseline does not know who trades with who. For example, if the Borland shock from D1 was passed into the baseline, only Borland's embedding would change and Aridia, Galenia, and Freland would stay the same since nothing connects them to Borland. The baseline here is also untrained so its weights are still random and its embeddings do not mean anything yet. To compare it with the GNN properly, both would need to be trained on the same objective (sigmoid(ZZ^T) with BCE on A_train) and then compared using things like the final loss and the ranks of the withheld links.

### Part E

In [11]:
A_train = torch.tensor([
    #A  B  C  D  E  F  G  H
    [0, 1, 1, 0, 0, 0, 0, 0], # A
    [1, 0, 0, 0, 0, 1, 1, 0], # B
    [1, 0, 0, 0, 0, 1, 1, 0], # C
    [0, 0, 0, 0, 1, 1, 0, 0], # D
    [0, 0, 0, 1, 0, 0, 0, 1], # E
    [0, 1, 1, 1, 0, 0, 1, 1], # F
    [0, 1, 1, 0, 0, 1, 0, 0], # G
    [0, 0, 0, 0, 1, 1, 0, 0] # H
    ], dtype=torch.float32
)

def aggregate_symmetric(A, H):
    I = torch.eye(A.shape[0])
    A_hat = A + I
    degree_hat = A_hat.sum(dim=1)
    degree_hat = degree_hat.clamp(min=1) # handle degrees safely, basically does max(degree_hat, 1)
    d_inv_sqrt = torch.pow(degree_hat, -0.5)
    d_inv_sqrt[torch.isinf(d_inv_sqrt)] = 0.0
    D_inv_sqrt = torch.diag(d_inv_sqrt)
    X_aggregated = D_inv_sqrt @ A_hat @ D_inv_sqrt @ H
    assert X_aggregated.shape == H.shape, "X_aggregated's shape == H.shape"

    return X_aggregated

class SymmetricGNNEncoder(nn.Module):
    def __init__(self, input_features=5, hidden_size=8, embedding_size=3):
        super().__init__()
        self.linear_one = nn.Linear(input_features, hidden_size)
        self.activation = nn.ReLU()
        self.linear_two = nn.Linear(hidden_size, embedding_size)

    def forward(self, A, X):
        layer_one = self.linear_one(aggregate_symmetric(A, X))
        X_activated = self.activation(layer_one)
        layer_two = self.linear_two(aggregate_symmetric(A, X_activated))
        return layer_two

class SymmetricGraphAutoencoder(nn.Module):
    def __init__(self, input_features_size, hidden_size, embedding_size):
        super().__init__()
        self.encoder = SymmetricGNNEncoder(input_features_size, hidden_size, embedding_size)

    def forward(self, A, X):
        Z = self.encoder(A, X)
        scores = Z @ Z.T
        A_reconstructed = torch.sigmoid(scores)
        return A_reconstructed, Z

torch.manual_seed(42)
INPUT_FEATURES = X_scaled
ENCODER_GRAPH = A_train
BASELINE_RECONSTRUCTION_TARGET = A_train
HIDDEN_DIMENSION = 8
EMBEDDING_DIMENSION = 3
LEARNING_RATE = 0.01
EPOCHS = 2000

model_new = SymmetricGraphAutoencoder(INPUT_FEATURES.shape[1], HIDDEN_DIMENSION, EMBEDDING_DIMENSION)
loss_function = nn.BCELoss()
optimizer = torch.optim.Adam(
    model_new.parameters(), lr=LEARNING_RATE
)

print("\n=== Part E: Training (symmetric normalization) ===")
for epoch in range(EPOCHS):
    optimizer.zero_grad()
    A_reconstructed, Z = model_new(ENCODER_GRAPH, INPUT_FEATURES)
    loss = loss_function(A_reconstructed, BASELINE_RECONSTRUCTION_TARGET)
    loss.backward()
    optimizer.step()

    if epoch % 200 == 0:
        print(f"Epoch {epoch}: loss = {loss}")

    if epoch == EPOCHS - 1:
        print(f"Final Loss: {loss}\n")

report(model_new, ENCODER_GRAPH, INPUT_FEATURES)


=== Part E: Training (symmetric normalization) ===
Epoch 0: loss = 0.7343781590461731
Epoch 200: loss = 0.5975755453109741
Epoch 400: loss = 0.5964644551277161
Epoch 600: loss = 0.5960640907287598
Epoch 800: loss = 0.5959018468856812
Epoch 1000: loss = 0.5957601070404053
Epoch 1200: loss = 0.5890806913375854
Epoch 1400: loss = 0.5825511813163757
Epoch 1600: loss = 0.5820570588111877


Epoch 1800: loss = 0.5817194581031799


Final Loss: 0.5815079808235168

Reconstructed edge probabilities (diagonal NOT forced to zero):
tensor([[0.8346, 0.6562, 0.7237, 0.2635, 0.2357, 0.3343, 0.5772, 0.2332],
        [0.6562, 0.7497, 0.7315, 0.2897, 0.1791, 0.6714, 0.7690, 0.3313],
        [0.7237, 0.7315, 0.7364, 0.2750, 0.1862, 0.5790, 0.7269, 0.2942],
        [0.2635, 0.2897, 0.2750, 0.7181, 0.7954, 0.4661, 0.3052, 0.7084],
        [0.2357, 0.1791, 0.1862, 0.7954, 0.8981, 0.3227, 0.1709, 0.7595],
        [0.3343, 0.6714, 0.5790, 0.4661, 0.3227, 0.8116, 0.7575, 0.5595],
        [0.5772, 0.7690, 0.7269, 0.3052, 0.1709, 0.7575, 0.8092, 0.3727],
        [0.2332, 0.3313, 0.2942, 0.7084, 0.7595, 0.5595, 0.3727, 0.7185]])

Country: Embedding
Aridia: tensor([-1.2610,  0.1636, -0.0389])
Borland: tensor([-0.4480,  0.6593,  0.6796])
Cyrenia: tensor([-0.7091,  0.5353,  0.4879])
Deltora: tensor([ 0.7688, -0.4479, -0.3783])
Estovia: tensor([ 0.8470, -0.8636, -0.8446])
Freland: tensor([0.6024, 0.6316, 0.8359])
Galenia: tensor([-0.1712,

(tensor([[0.8346, 0.6562, 0.7237, 0.2635, 0.2357, 0.3343, 0.5772, 0.2332],
         [0.6562, 0.7497, 0.7315, 0.2897, 0.1791, 0.6714, 0.7690, 0.3313],
         [0.7237, 0.7315, 0.7364, 0.2750, 0.1862, 0.5790, 0.7269, 0.2942],
         [0.2635, 0.2897, 0.2750, 0.7181, 0.7954, 0.4661, 0.3052, 0.7084],
         [0.2357, 0.1791, 0.1862, 0.7954, 0.8981, 0.3227, 0.1709, 0.7595],
         [0.3343, 0.6714, 0.5790, 0.4661, 0.3227, 0.8116, 0.7575, 0.5595],
         [0.5772, 0.7690, 0.7269, 0.3052, 0.1709, 0.7575, 0.8092, 0.3727],
         [0.2332, 0.3313, 0.2942, 0.7084, 0.7595, 0.5595, 0.3727, 0.7185]]),
 tensor([[-1.2610,  0.1636, -0.0389],
         [-0.4480,  0.6593,  0.6796],
         [-0.7091,  0.5353,  0.4879],
         [ 0.7688, -0.4479, -0.3783],
         [ 0.8470, -0.8636, -0.8446],
         [ 0.6024,  0.6316,  0.8359],
         [-0.1712,  0.7941,  0.8861],
         [ 0.9126, -0.2806, -0.1596]]))

In [12]:
# E comparison
# same checks as C1-C3, but this time on model_new so we can compare it with the original model
print("\n=== Part E: Comparison with the original model ===")
with torch.no_grad():
    probabilities_new, Z_new = model_new(ENCODER_GRAPH, INPUT_FEATURES)

print(f"Aridia - Galenia: {probabilities_new[aridia, galenia]}")
print(f"Aridia - Estovia: {probabilities_new[aridia, estovia]}")
print(f"Deltora - Hesperia: {probabilities_new[deltora, hesperia]}")
print(f"Deltora - Cyrenia: {probabilities_new[deltora, cyrenia]}\n")

rows, cols = unique_pairs(A_train)
reconstructed_scores_new = []
for i, j in zip(rows.tolist(), cols.tolist()):
    if A_train[i, j] == 0:
        reconstructed_scores_new.append((probabilities_new[i, j], i, j))
reconstructed_scores_new = sorted(reconstructed_scores_new, key=lambda x:x[0], reverse=True)

for idx, score in enumerate(reconstructed_scores_new):
    if score[1] == aridia and score[2] == galenia:
        print(f"Aridia - Galenia is Rank {idx + 1}")
    if score[1] == deltora and score[2] == hesperia:
        print(f"Deltora - Hesperia is Rank {idx + 1}")
print()

distances_new = torch.cdist(Z_new, Z_new)
for i in range(len(names)):
    closest = float('inf')
    for j in range(len(names)):
        if i == j: continue
        if distances_new[i, j] < closest:
            closest = distances_new[i, j]
            closest_country = j
    print(f"The closest country to {names[i]} is {names[closest_country]}. Distance: {closest:.4f}")

centroid_new = Z_new.mean(dim=0)
centroid_distances_new = torch.norm(Z_new - centroid_new, dim=1)
anomaly_ranking_new = sorted(
    range(len(names)), key=lambda i: centroid_distances_new[i].item(), reverse=True
)

print("\nRank  Country     Distance to centroid")
for rank, i in enumerate(anomaly_ranking_new, start=1):
    print(f"{rank:<5} {names[i]:<11} {centroid_distances_new[i]:.4f}")


=== Part E: Comparison with the original model ===
Aridia - Galenia: 0.5772046446800232
Aridia - Estovia: 0.23568251729011536
Deltora - Hesperia: 0.7084121108055115
Deltora - Cyrenia: 0.27498868107795715

Deltora - Hesperia is Rank 2
Aridia - Galenia is Rank 3

The closest country to Aridia is Cyrenia. Distance: 0.8486
The closest country to Borland is Cyrenia. Distance: 0.3469
The closest country to Cyrenia is Borland. Distance: 0.3469
The closest country to Deltora is Hesperia. Distance: 0.3106
The closest country to Estovia is Deltora. Distance: 0.6296
The closest country to Freland is Galenia. Distance: 0.7921
The closest country to Galenia is Borland. Distance: 0.3707
The closest country to Hesperia is Deltora. Distance: 0.3106

Rank  Country     Distance to centroid
1     Estovia     1.6400
2     Aridia      1.3472
3     Deltora     1.0786
4     Hesperia    1.0080
5     Galenia     0.9833
6     Freland     0.9719
7     Cyrenia     0.9194
8     Borland     0.8789


We can observe that the model that used symmetric normalization (0.58) has a lower final loss compared to the model that used row-mean aggregation (0.59) so it is better at reconstructing the given training data. Comparing the predictions of the two models on the link between the withheld links, the original model showed A:G(0.7180) and D:H(0.6767) and the new model showed A:G(0.5772) and D:H(0.7084), we can see that the original model is more confident in A and G linkage but the new model is more confident in the link between D and H but less confident in the other. In terms of ranks, the original model had A:G at rank 2 and D:H at rank 3, while the new model swapped them, with D:H at rank 2 and A:G at rank 3. The nearest neighbors stayed almost the same, and the only change is Freland, whose closest country went from Borland to Galenia. For the anomaly ranking, Estovia is still the most unusual country in both models (1.6319 vs 1.6400), but Aridia jumped from 6th to 2nd, and Freland went from being the closest to the centroid (8th) to 6th. A lower reconstruction loss does not necessarily mean better hidden-link rankings because these models are trained to reconstruct the original dataset which can harm the ratings between the withheld links.